# 01 - Data Profiling

This notebook loads the 5 raw CSV datasets from data/raw/ and performs initial data profiling:
- Schema and data types
- Head/tail preview
- Descriptive statistics
- Null counts and memory usage
- Data profile report from outputs/metrics/data_profile.json

## 1. Setup and Imports

In [ ]:
import pandas as pd
import numpy as np
import json
import os
from pathlib import Path

import warnings
warnings.filterwarnings("ignore")

print("Libraries loaded successfully.")

## 2. Load Raw Datasets

In [ ]:
raw_data_path = Path('../data/raw')

raw_data = {}

csv_files = ["orders.csv", "order_items.csv", "products.csv", "customers.csv", "inventory.csv"]

for file in csv_files:
    file_path = raw_data_path / file
    if file_path.exists():
        key = file.replace('.csv', '')
        raw_data[key] = pd.read_csv(file_path)
        print(f"Loaded {file}: {raw_data[key].shape[0]} rows, {raw_data[key].shape[1]} columns")
    else:
        print(f"WARNING: {file} not found at {file_path}")

print(f"\nTotal datasets loaded: {len(raw_data)}")

## 3. Schema and Data Types

In [ ]:
for name, df in raw_data.items():
    sep = "=" * 60
    print(f"\n{sep}\nSCHEMA: {name.upper()}\n{sep}")
    print(f"Shape: {df.shape}")
    print("\nData Types:")
    print(df.dtypes.to_string())

## 4. Head and Tail Preview

In [ ]:
for name, df in raw_data.items():
    sep = "=" * 60
    print(f"\n{sep}\nPREVIEW: {name.upper()}\n{sep}")
    print("\n--- First 5 rows ---")
    print(df.head().to_string())
    print("\n--- Last 5 rows ---")
    print(df.tail().to_string())

## 5. Descriptive Statistics

In [ ]:
for name, df in raw_data.items():
    sep = "=" * 60
    print(f"\n{sep}\nDESCRIPTIVE STATISTICS: {name.upper()}\n{sep}")
    print(df.describe(include="all").to_string())

## 6. Null Counts and Missing Data

In [ ]:
for name, df in raw_data.items():
    sep = "=" * 60
    print(f"\n{sep}\nNULL ANALYSIS: {name.upper()}\n{sep}")
    null_counts = df.isnull().sum()
    null_pct = (df.isnull().sum() / len(df) * 100).round(2)
    null_df = pd.DataFrame({"Null Count": null_counts, "Null %": null_pct})
    null_df = null_df[null_df["Null Count"] > 0].sort_values("Null %", ascending=False)
    if len(null_df) > 0:
        print(null_df.to_string())
    else:
        print("No missing values found.")

## 7. Memory Usage

In [ ]:
memory_summary = {}

for name, df in raw_data.items():
    mem_bytes = df.memory_usage(deep=True).sum()
    mem_mb = mem_bytes / (1024 * 1024)
    memory_summary[name] = {'bytes': mem_bytes, 'mb': round(mem_mb, 2)}
    print(f"{name}: {mem_mb:.2f} MB")

total_memory = sum(v["mb"] for v in memory_summary.values())
print(f"\nTotal memory usage: {total_memory:.2f} MB")

## 8. Data Profile Report

In [ ]:
profile_path = Path('../outputs/metrics/data_profile.json')

if profile_path.exists():
    with open(profile_path, "r") as f:
        profile = json.load(f)
    print("DATA PROFILE REPORT")
    print("=" * 60)
    print(json.dumps(profile, indent=2, default=str))
else:
    print(f"Profile report not found at {profile_path}")
    print("Run the data profiling script first.")

## 9. Summary

In [ ]:
summary_rows = []
for name, df in raw_data.items():
    summary_rows.append({
        "Dataset": name,
        "Rows": df.shape[0],
        "Columns": df.shape[1],
        "Null Columns": df.isnull().any().sum(),
        "Memory (MB)": memory_summary[name]["mb"]
    })

summary_df = pd.DataFrame(summary_rows)
print("DATASET SUMMARY")
print("=" * 60)
print(summary_df.to_string(index=False) if len(summary_rows) > 0 else "No data loaded.")